# Global Automotive Investment Database — Block 1

## ETF constituent retrieval and point-in-time holdings reconstruction

**Universe:** DRIV, CARZ, IDRV and KARS  
**Primary source:** official SEC Form N-PORT structured-data bulk files  
**Core outputs:** pandas DataFrames plus persistent Parquet hand-off files for Block 2

This revised version preserves the original point-in-time reconstruction logic while changing the execution architecture:

1. SEC quarterly ZIP files are cached persistently in Google Drive.
2. Block 1 outputs are written to a stable `data/interim/block_1/` folder.
3. A manifest records row counts, columns, file paths and creation timestamps.
4. Later notebooks load the persisted outputs directly and do not rerun Block 1.

Created: July 2026


In [ ]:
# 1. INSTALL REQUIRED PACKAGES
# ------------------------------------------------

!pip -q install pandas numpy requests beautifulsoup4 lxml tqdm pyarrow


# ------------------------------------------------


# 2. IMPORTS
# ------------------------------------------------

from __future__ import annotations

import json
import re
import time
import zipfile

from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests

from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from google.colab import userdata

def get_secret(name):
    try:
        return userdata.get(name).strip()
    except Exception:
        raise ValueError(
            f"Create a Colab secret called '{name}'."
        )

# ------------------------------------------------


# 3. USER SETTINGS
# ------------------------------------------------

# SEC automated-access identification.
SEC_USER_AGENT = get_secret("SEC_USER_AGENT")

# Earliest publicly available systematic N-PORT bulk dataset.
START_YEAR = 2019
START_QUARTER = 4

# Leave both as None to process every currently available quarter.
END_YEAR = None
END_QUARTER = None

# Reuse quarterly ZIP files already downloaded to Google Drive.
USE_CACHE = True

# Block 1 now uses persistent Google Drive storage by default.
USE_GOOGLE_DRIVE = True

# Remove obvious cash, repo, debt and derivative positions.
KEEP_NON_EQUITY_HOLDINGS = False

# For a quick test, set to an integer such as 2.
# For the complete historical reconstruction, leave as None.
MAX_QUARTERS_TO_PROCESS = None

# SEC request delay.
SEC_REQUEST_DELAY_SECONDS = 0.15

# Persist the completed Block 1 hand-off tables for later blocks.
PERSIST_BLOCK_1_OUTPUTS = True

# Overwrite existing Block 1 Parquet outputs.
OVERWRITE_PERSISTED_OUTPUTS = True


# ------------------------------------------------


# 4. ETF LEGAL-NAME ALIASES
# ------------------------------------------------

TARGET_FUNDS = {

    "DRIV": [
        "Global X Autonomous & Electric Vehicles ETF",
        "Global X Autonomous and Electric Vehicles ETF",
        "Autonomous & Electric Vehicles ETF",
        "Autonomous and Electric Vehicles ETF",
    ],

    "CARZ": [
        "First Trust S-Network Future Vehicles & Technology ETF",
        "First Trust S-Network Future Vehicles and Technology ETF",
        "S-Network Future Vehicles & Technology ETF",
        "S-Network Future Vehicles and Technology ETF",
        "First Trust NASDAQ Global Auto Index Fund",
        "First Trust Nasdaq Global Auto Index Fund",
        "NASDAQ Global Auto Index Fund",
        "Nasdaq Global Auto Index Fund",
    ],

    "IDRV": [
        "iShares Self-Driving EV and Tech ETF",
        "iShares Self Driving EV and Tech ETF",
        "Self-Driving EV and Tech ETF",
        "Self Driving EV and Tech ETF",
    ],

    "KARS": [
        "KraneShares Electric Vehicles and Future Mobility Index ETF",
        "KraneShares Electric Vehicles & Future Mobility ETF",
        "KraneShares Electric Vehicles and Future Mobility ETF",
        "Electric Vehicles and Future Mobility Index ETF",
        "Electric Vehicles and Future Mobility ETF",
    ],
}


# ------------------------------------------------


# 5. DIRECTORY SETUP
# ------------------------------------------------

if USE_GOOGLE_DRIVE:

    from google.colab import drive

    drive.mount("/content/drive")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy"
    )

else:

    PROJECT_ROOT = Path("/content/global_automotive_investment_database")


DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw" / "sec_nport"
CACHE_DIR = RAW_DATA_DIR / "quarterly_zip_cache"
INTERIM_DATA_DIR = DATA_ROOT / "interim"
BLOCK_1_OUTPUT_DIR = INTERIM_DATA_DIR / "block_1"
BLOCK_1_MANIFEST_PATH = BLOCK_1_OUTPUT_DIR / "block_1_manifest.json"

for directory in [
    PROJECT_ROOT,
    DATA_ROOT,
    RAW_DATA_DIR,
    CACHE_DIR,
    INTERIM_DATA_DIR,
    BLOCK_1_OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

BASE_DIR = DATA_ROOT

print("Project root:", PROJECT_ROOT)
print("N-PORT cache:", CACHE_DIR)
print("Block 1 outputs:", BLOCK_1_OUTPUT_DIR)


# ------------------------------------------------


# 6. VALIDATE SEC USER AGENT
# ------------------------------------------------

invalid_user_agent = (

    "@" not in SEC_USER_AGENT

    or "example.com" in SEC_USER_AGENT.lower()

    or "your-real-email" in SEC_USER_AGENT.lower()
)

if invalid_user_agent:

    raise ValueError(
        "Replace SEC_USER_AGENT with your real name and email address."
    )


# ------------------------------------------------


# 7. GENERAL HELPERS
# ------------------------------------------------

def normalise_text(value) -> str:
    """
    Standardise names for tolerant legal-name matching.
    """

    if pd.isna(value):
        return ""

    text = str(value).casefold()

    text = text.replace(
        "&",
        " and ",
    )

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def clean_columns(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Standardise SEC flat-file column names.
    """

    output = dataframe.copy()

    output.columns = [

        re.sub(
            r"[^A-Z0-9_]+",
            "_",
            str(column).strip().upper(),
        ).strip("_")

        for column in output.columns
    ]

    return output


def first_existing_column(
    dataframe: pd.DataFrame,
    possibilities: list[str],
) -> str | None:
    """
    Return the first candidate column found in a DataFrame.
    """

    for column in possibilities:

        if column in dataframe.columns:
            return column

    return None


def coalesce_columns(
    dataframe: pd.DataFrame,
    possibilities: list[str],
) -> pd.Series:
    """
    Combine multiple possible source columns into one Series.
    """

    result = pd.Series(
        pd.NA,
        index=dataframe.index,
        dtype="string",
    )

    for column in possibilities:

        if column in dataframe.columns:

            result = result.fillna(
                dataframe[column].astype("string")
            )

    return result


def first_non_missing(
    series: pd.Series,
):
    """
    Return the first non-empty value in a Series.
    """

    values = (

        series
        .dropna()
        .astype("string")
        .str.strip()
    )

    values = values[
        values.ne("")
    ]

    if values.empty:
        return pd.NA

    return values.iloc[0]


def quarter_number(
    year: int,
    quarter: int,
) -> int:
    """
    Convert year and quarter to a sortable integer.
    """

    return year * 4 + quarter


def is_quarter_in_range(
    year: int,
    quarter: int,
) -> bool:
    """
    Apply configured historical start and end dates.
    """

    current = quarter_number(
        year,
        quarter,
    )

    start = quarter_number(
        START_YEAR,
        START_QUARTER,
    )

    if current < start:
        return False

    if (
        END_YEAR is not None
        and END_QUARTER is not None
    ):

        end = quarter_number(
            END_YEAR,
            END_QUARTER,
        )

        if current > end:
            return False

    return True


def safe_numeric(
    series: pd.Series,
) -> pd.Series:
    """
    Convert SEC numeric fields robustly.
    """

    return pd.to_numeric(

        series
        .astype("string")
        .str.replace(
            ",",
            "",
            regex=False,
        )
        .str.replace(
            "%",
            "",
            regex=False,
        ),

        errors="coerce",
    )


# ------------------------------------------------


Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy
N-PORT cache: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/raw/sec_nport/quarterly_zip_cache
Block 1 outputs: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_1


In [ ]:
# 8. SEC HTTP SESSION
# ------------------------------------------------

session = requests.Session()

session.headers.update({

    "User-Agent":
        SEC_USER_AGENT,

    "Accept-Encoding":
        "gzip, deflate",

})


def sec_get(
    url: str,
    *,
    stream: bool = False,
    timeout: int = 120,
    retries: int = 5,
):
    """
    SEC request helper with delay and retry logic.
    """

    last_error = None

    for attempt in range(retries):

        try:

            time.sleep(
                SEC_REQUEST_DELAY_SECONDS
            )

            response = session.get(
                url,
                stream=stream,
                timeout=timeout,
            )

            if response.status_code == 404:
                raise FileNotFoundError(url)

            if response.status_code in {
                403,
                429,
                500,
                502,
                503,
                504,
            }:

                wait_seconds = 2 ** attempt

                print(
                    f"SEC returned {response.status_code}. "
                    f"Retrying in {wait_seconds} seconds."
                )

                time.sleep(
                    wait_seconds
                )

                continue

            response.raise_for_status()

            return response

        except FileNotFoundError:
            raise

        except Exception as error:

            last_error = error

            if attempt == retries - 1:
                break

            time.sleep(
                2 ** attempt
            )

    raise RuntimeError(
        f"Failed to retrieve:\n{url}\n\n"
        f"Last error: {last_error}"
    )


# ------------------------------------------------


# 9. DISCOVER AVAILABLE N-PORT DATASETS
# ------------------------------------------------

NPORT_DATASET_PAGE = (
    "https://www.sec.gov/data-research/"
    "sec-markets-data/form-n-port-data-sets"
)


def discover_nport_downloads() -> pd.DataFrame:
    """
    Discover all quarterly N-PORT ZIP links from the SEC page.
    """

    response = sec_get(
        NPORT_DATASET_PAGE
    )

    soup = BeautifulSoup(
        response.content,
        "html.parser",
    )

    rows = []

    for link in soup.find_all(
        "a",
        href=True,
    ):

        href = link["href"]

        match = re.search(
            r"(\d{4})q([1-4])_nport\.zip",
            href,
            flags=re.IGNORECASE,
        )

        if match is None:
            continue

        year = int(
            match.group(1)
        )

        quarter = int(
            match.group(2)
        )

        if not is_quarter_in_range(
            year,
            quarter,
        ):
            continue

        rows.append({

            "year":
                year,

            "quarter":
                quarter,

            "url":
                urljoin(
                    NPORT_DATASET_PAGE,
                    href,
                ),

        })

    datasets = pd.DataFrame(
        rows
    )

    if datasets.empty:

        raise RuntimeError(
            "No N-PORT datasets were found on the SEC page."
        )

    datasets = (

        datasets
        .drop_duplicates(
            [
                "year",
                "quarter",
            ]
        )
        .sort_values(
            [
                "year",
                "quarter",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    if MAX_QUARTERS_TO_PROCESS is not None:

        datasets = (

            datasets
            .tail(
                int(
                    MAX_QUARTERS_TO_PROCESS
                )
            )
            .reset_index(
                drop=True
            )
        )

    return datasets


available_datasets = (
    discover_nport_downloads()
)


# ------------------------------------------------


# 10. DOWNLOAD A QUARTERLY N-PORT ZIP
# ------------------------------------------------

def download_quarter(
    year: int,
    quarter: int,
    url: str,
) -> Path:
    """
    Download and cache one quarterly SEC N-PORT dataset.
    """

    destination = (
        CACHE_DIR
        / f"{year}q{quarter}_nport.zip"
    )

    valid_cached_file = (

        USE_CACHE

        and destination.exists()

        and destination.stat().st_size > 10_000
    )

    if valid_cached_file:
        return destination

    temporary_path = destination.with_suffix(
        ".zip.part"
    )

    response = sec_get(
        url,
        stream=True,
        timeout=300,
    )

    total_bytes = int(
        response.headers.get(
            "content-length",
            0,
        )
    )

    with (

        temporary_path.open(
            "wb"
        ) as output_file,

        tqdm(
            total=total_bytes,
            unit="B",
            unit_scale=True,
            desc=f"{year} Q{quarter}",
            leave=False,
        ) as progress_bar,

    ):

        for chunk in response.iter_content(
            chunk_size=1024 * 1024
        ):

            if not chunk:
                continue

            output_file.write(
                chunk
            )

            progress_bar.update(
                len(chunk)
            )

    temporary_path.replace(
        destination
    )

    return destination


# ------------------------------------------------


# 11. READ TABLES FROM AN N-PORT ZIP
# ------------------------------------------------

def find_archive_member(
    archive: zipfile.ZipFile,
    possible_names: Iterable[str],
) -> str:
    """
    Find a requested table inside the SEC ZIP archive.
    """

    archive_names = archive.namelist()

    exact_lookup = {

        Path(name).name.casefold():
            name

        for name in archive_names
    }

    for possible_name in possible_names:

        key = possible_name.casefold()

        if key in exact_lookup:
            return exact_lookup[key]

    possible_stems = {

        Path(name).stem.casefold()

        for name in possible_names
    }

    for archive_name in archive_names:

        archive_stem = (
            Path(archive_name)
            .stem
            .casefold()
        )

        if archive_stem in possible_stems:
            return archive_name

    raise FileNotFoundError(
        f"Could not find {list(possible_names)} inside archive."
    )


def read_archive_table(
    archive: zipfile.ZipFile,
    possible_names: list[str],
) -> pd.DataFrame:
    """
    Read a tab-separated SEC N-PORT table.
    """

    member_name = find_archive_member(
        archive,
        possible_names,
    )

    with archive.open(
        member_name
    ) as file_handle:

        dataframe = pd.read_csv(
            file_handle,
            sep="\t",
            dtype="string",
            low_memory=False,
            na_values=[
                "",
                "N/A",
                "NA",
                "NULL",
                "null",
            ],
        )

    return clean_columns(
        dataframe
    )


# ------------------------------------------------


In [ ]:
# 12. BUILD ETF ALIAS TABLE
# ------------------------------------------------

def build_alias_table() -> pd.DataFrame:
    """
    Convert TARGET_FUNDS into a matchable alias table.
    """

    rows = []

    for etf, aliases in TARGET_FUNDS.items():

        for alias in aliases:

            rows.append({

                "etf":
                    etf,

                "alias":
                    alias,

                "alias_normalised":
                    normalise_text(alias),

            })

    return pd.DataFrame(
        rows
    )


fund_aliases = build_alias_table()


# ------------------------------------------------


# 13. IDENTIFY TARGET FUND FILINGS
# ------------------------------------------------

def identify_target_filings(
    fund_info: pd.DataFrame,
) -> pd.DataFrame:
    """
    Match SEC series names against target ETF aliases.
    """

    accession_column = first_existing_column(
        fund_info,
        [
            "ACCESSION_NUMBER",
            "ACCESSION_NO",
        ],
    )

    series_name_column = first_existing_column(
        fund_info,
        [
            "SERIES_NAME",
            "FUND_NAME",
        ],
    )

    series_id_column = first_existing_column(
        fund_info,
        [
            "SERIES_ID",
        ],
    )

    if accession_column is None:

        raise KeyError(
            "FUND_REPORTED_INFO has no accession-number column."
        )

    if series_name_column is None:

        raise KeyError(
            "FUND_REPORTED_INFO has no series-name column."
        )

    working = fund_info.copy()

    working[
        "SERIES_NAME_STANDARD"
    ] = (
        working[
            series_name_column
        ].astype("string")
    )

    working[
        "SERIES_NAME_NORMALISED"
    ] = (
        working[
            "SERIES_NAME_STANDARD"
        ]
        .map(
            normalise_text
        )
    )

    matches = []

    for alias_row in fund_aliases.itertuples(
        index=False
    ):

        alias = alias_row.alias_normalised

        exact_match = (
            working[
                "SERIES_NAME_NORMALISED"
            ].eq(alias)
        )

        contained_match = (
            working[
                "SERIES_NAME_NORMALISED"
            ].str.contains(
                re.escape(alias),
                regex=True,
                na=False,
            )
        )

        mask = (
            exact_match
            | contained_match
        )

        if not mask.any():
            continue

        selected_columns = [
            accession_column,
            "SERIES_NAME_STANDARD",
        ]

        if series_id_column is not None:

            selected_columns.append(
                series_id_column
            )

        matched = working.loc[
            mask,
            selected_columns,
        ].copy()

        rename_mapping = {

            accession_column:
                "ACCESSION_NUMBER",

        }

        if series_id_column is not None:

            rename_mapping[
                series_id_column
            ] = "SERIES_ID"

        matched = matched.rename(
            columns=rename_mapping
        )

        if "SERIES_ID" not in matched.columns:

            matched[
                "SERIES_ID"
            ] = pd.NA

        matched[
            "etf"
        ] = alias_row.etf

        matched[
            "matched_alias"
        ] = alias_row.alias

        matched[
            "alias_length"
        ] = len(
            alias_row.alias_normalised
        )

        matches.append(
            matched
        )

    if not matches:

        return pd.DataFrame(
            columns=[
                "ACCESSION_NUMBER",
                "SERIES_NAME_STANDARD",
                "SERIES_ID",
                "etf",
                "matched_alias",
            ]
        )

    result = pd.concat(
        matches,
        ignore_index=True,
    )

    result = (

        result
        .sort_values(
            "alias_length",
            ascending=False,
        )
        .drop_duplicates(
            [
                "ACCESSION_NUMBER",
                "etf",
            ],
            keep="first",
        )
        .drop(
            columns="alias_length"
        )
    )

    return result


# ------------------------------------------------


# 14. COLLAPSE IDENTIFIER ROWS
# ------------------------------------------------

def collapse_identifiers(
    identifiers: pd.DataFrame,
) -> pd.DataFrame:
    """
    Collapse multiple identifier rows into one row per holding.
    """

    if identifiers.empty:
        return identifiers

    holding_id_column = first_existing_column(
        identifiers,
        [
            "HOLDING_ID",
        ],
    )

    if holding_id_column is None:
        return pd.DataFrame()

    result = pd.DataFrame({

        "HOLDING_ID":
            identifiers[
                holding_id_column
            ].astype("string"),

        "IDENTIFIER_ISIN":
            coalesce_columns(
                identifiers,
                [
                    "IDENTIFIER_ISIN",
                    "ISIN",
                ],
            ),

        "IDENTIFIER_TICKER":
            coalesce_columns(
                identifiers,
                [
                    "IDENTIFIER_TICKER",
                    "TICKER",
                ],
            ),

        "OTHER_IDENTIFIER":
            coalesce_columns(
                identifiers,
                [
                    "OTHER_IDENTIFIER",
                    "IDENTIFIER_OTHER",
                ],
            ),

        "OTHER_IDENTIFIER_DESC":
            coalesce_columns(
                identifiers,
                [
                    "OTHER_IDENTIFIER_DESC",
                    "OTHER_IDENTIFIER_DESCRIPTION",
                ],
            ),

    })

    result = (

        result
        .groupby(
            "HOLDING_ID",
            as_index=False,
            dropna=False,
        )
        .agg({

            "IDENTIFIER_ISIN":
                first_non_missing,

            "IDENTIFIER_TICKER":
                first_non_missing,

            "OTHER_IDENTIFIER":
                first_non_missing,

            "OTHER_IDENTIFIER_DESC":
                first_non_missing,

        })
    )

    return result


# ------------------------------------------------


# 15. PROCESS ONE QUARTER
# ------------------------------------------------

def process_quarter(
    zip_path: Path,
    year: int,
    quarter: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Extract target ETF filings and holdings from one quarter.
    """

    with zipfile.ZipFile(
        zip_path
    ) as archive:

        submission = read_archive_table(
            archive,
            [
                "SUBMISSION.tsv",
                "SUBMISSION.txt",
            ],
        )

        fund_info = read_archive_table(
            archive,
            [
                "FUND_REPORTED_INFO.tsv",
                "FUND_REPORTED_INFO.txt",
            ],
        )

        target_filings = identify_target_filings(
            fund_info
        )

        if target_filings.empty:

            return (
                pd.DataFrame(),
                pd.DataFrame(),
            )

        target_accessions = set(
            target_filings[
                "ACCESSION_NUMBER"
            ].dropna()
        )

        holdings = read_archive_table(
            archive,
            [
                "FUND_REPORTED_HOLDING.tsv",
                "FUND_REPORTED_HOLDING.txt",
            ],
        )

        holdings_accession_column = first_existing_column(
            holdings,
            [
                "ACCESSION_NUMBER",
                "ACCESSION_NO",
            ],
        )

        if holdings_accession_column is None:

            raise KeyError(
                "FUND_REPORTED_HOLDING has no accession-number column."
            )

        holdings = holdings[
            holdings[
                holdings_accession_column
            ].isin(
                target_accessions
            )
        ].copy()

        holdings = holdings.rename(
            columns={
                holdings_accession_column:
                    "ACCESSION_NUMBER"
            }
        )

        try:

            identifiers = read_archive_table(
                archive,
                [
                    "IDENTIFIERS.tsv",
                    "IDENTIFIERS.txt",
                ],
            )

            if (
                "HOLDING_ID" in holdings.columns
                and "HOLDING_ID" in identifiers.columns
            ):

                target_holding_ids = set(
                    holdings[
                        "HOLDING_ID"
                    ].dropna()
                )

                identifiers = identifiers[
                    identifiers[
                        "HOLDING_ID"
                    ].isin(
                        target_holding_ids
                    )
                ].copy()

                identifiers = collapse_identifiers(
                    identifiers
                )

                if not identifiers.empty:

                    holdings = holdings.merge(
                        identifiers,
                        on="HOLDING_ID",
                        how="left",
                    )

        except FileNotFoundError:
            pass

    submission_accession_column = first_existing_column(
        submission,
        [
            "ACCESSION_NUMBER",
            "ACCESSION_NO",
        ],
    )

    if submission_accession_column is not None:

        submission_metadata = submission.rename(
            columns={
                submission_accession_column:
                    "ACCESSION_NUMBER"
            }
        ).copy()

        useful_submission_columns = [

            column

            for column in [
                "ACCESSION_NUMBER",
                "FILING_DATE",
                "SUB_TYPE",
                "REPORT_DATE",
                "REPORT_ENDING_PERIOD",
                "PERIOD_END",
            ]

            if column in submission_metadata.columns
        ]

        submission_metadata = (

            submission_metadata[
                useful_submission_columns
            ]
            .drop_duplicates(
                "ACCESSION_NUMBER"
            )
        )

        target_filings = target_filings.merge(
            submission_metadata,
            on="ACCESSION_NUMBER",
            how="left",
        )

    target_filings[
        "dataset_year"
    ] = year

    target_filings[
        "dataset_quarter"
    ] = quarter

    holdings = holdings.merge(
        target_filings,
        on="ACCESSION_NUMBER",
        how="left",
        suffixes=(
            "",
            "_FUND",
        ),
    )

    holdings[
        "dataset_year"
    ] = year

    holdings[
        "dataset_quarter"
    ] = quarter

    return (
        target_filings,
        holdings,
    )


# ------------------------------------------------


In [ ]:
# 16. STANDARDISE HOLDINGS
# ------------------------------------------------

def standardise_holdings(
    raw_holdings: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert SEC fields into a stable research schema.
    """

    if raw_holdings.empty:
        return raw_holdings.copy()

    dataframe = raw_holdings.copy()

    source_mapping = {

        "ACCESSION_NUMBER":
            "accession_number",

        "HOLDING_ID":
            "holding_id",

        "SERIES_NAME_STANDARD":
            "series_name",

        "SERIES_ID":
            "series_id",

        "FILING_DATE":
            "filing_date",

        "REPORT_DATE":
            "snapshot_date",

        "REPORT_ENDING_PERIOD":
            "fiscal_period_end",

        "PERIOD_END":
            "fiscal_period_end_alt",

        "SUB_TYPE":
            "submission_type",

        "ISSUER_NAME":
            "issuer_name",

        "ISSUER_TITLE":
            "security_title",

        "ISSUER_CUSIP":
            "cusip",

        "CUSIP":
            "cusip_alt",

        "ISSUER_LEI":
            "issuer_lei",

        "IDENTIFIER_ISIN":
            "isin",

        "IDENTIFIER_TICKER":
            "ticker",

        "OTHER_IDENTIFIER":
            "other_identifier",

        "OTHER_IDENTIFIER_DESC":
            "other_identifier_description",

        "BALANCE":
            "balance",

        "UNIT":
            "unit",

        "CURRENCY_CODE":
            "currency",

        "CURRENCY_VALUE":
            "market_value",

        "VALUE_USD":
            "market_value_usd",

        "PERCENTAGE":
            "reported_weight",

        "PAYOFF_PROFILE":
            "payoff_profile",

        "ASSET_CAT":
            "asset_category",

        "ISSUER_TYPE":
            "issuer_type",

        "INVESTMENT_COUNTRY":
            "investment_country",

        "IS_RESTRICTED_SECURITY":
            "is_restricted",

        "FAIR_VALUE_LEVEL":
            "fair_value_level",

    }

    dataframe = dataframe.rename(
        columns={

            source:
                target

            for source, target in source_mapping.items()

            if source in dataframe.columns
        }
    )

    if "ticker" not in dataframe.columns:

        dataframe[
            "ticker"
        ] = pd.NA

    for source_column in [
        "TICKER",
        "ISSUER_TICKER",
    ]:

        if source_column in dataframe.columns:

            dataframe[
                "ticker"
            ] = (
                dataframe[
                    "ticker"
                ]
                .astype("string")
                .fillna(
                    dataframe[
                        source_column
                    ].astype("string")
                )
            )

    if "cusip" not in dataframe.columns:

        dataframe[
            "cusip"
        ] = pd.NA

    if "cusip_alt" in dataframe.columns:

        dataframe[
            "cusip"
        ] = (
            dataframe[
                "cusip"
            ]
            .astype("string")
            .fillna(
                dataframe[
                    "cusip_alt"
                ].astype("string")
            )
        )

    if "snapshot_date" not in dataframe.columns:

        dataframe[
            "snapshot_date"
        ] = pd.NaT

    dataframe[
        "snapshot_date"
    ] = pd.to_datetime(
        dataframe[
            "snapshot_date"
        ],
        errors="coerce",
    )

    if "fiscal_period_end" in dataframe.columns:

        dataframe[
            "snapshot_date"
        ] = dataframe[
            "snapshot_date"
        ].fillna(

            pd.to_datetime(
                dataframe[
                    "fiscal_period_end"
                ],
                errors="coerce",
            )
        )

    if "fiscal_period_end_alt" in dataframe.columns:

        dataframe[
            "snapshot_date"
        ] = dataframe[
            "snapshot_date"
        ].fillna(

            pd.to_datetime(
                dataframe[
                    "fiscal_period_end_alt"
                ],
                errors="coerce",
            )
        )

    for date_column in [
        "filing_date",
        "snapshot_date",
        "fiscal_period_end",
        "fiscal_period_end_alt",
    ]:

        if date_column in dataframe.columns:

            dataframe[
                date_column
            ] = pd.to_datetime(
                dataframe[
                    date_column
                ],
                errors="coerce",
            )

    for numeric_column in [
        "balance",
        "market_value",
        "market_value_usd",
        "reported_weight",
    ]:

        if numeric_column in dataframe.columns:

            dataframe[
                numeric_column
            ] = safe_numeric(
                dataframe[
                    numeric_column
                ]
            )

    if "market_value_usd" not in dataframe.columns:

        dataframe[
            "market_value_usd"
        ] = np.nan

    if "market_value" in dataframe.columns:

        dataframe[
            "market_value_usd"
        ] = (
            pd.to_numeric(
                dataframe[
                    "market_value_usd"
                ],
                errors="coerce",
            )
            .fillna(
                pd.to_numeric(
                    dataframe[
                        "market_value"
                    ],
                    errors="coerce",
                )
            )
        )

    if "reported_weight" in dataframe.columns:

        dataframe[
            "weight"
        ] = (
            dataframe[
                "reported_weight"
            ]
            / 100
        )

    else:

        dataframe[
            "weight"
        ] = np.nan

    dataframe[
        "effective_date"
    ] = dataframe[
        "filing_date"
    ]

    dataframe[
        "information_lag_days"
    ] = (

        dataframe[
            "effective_date"
        ]

        - dataframe[
            "snapshot_date"
        ]

    ).dt.days

    for identifier_column in [
        "ticker",
        "cusip",
        "isin",
        "issuer_lei",
    ]:

        if identifier_column not in dataframe.columns:

            dataframe[
                identifier_column
            ] = pd.NA

        dataframe[
            identifier_column
        ] = (

            dataframe[
                identifier_column
            ]
            .astype("string")
            .str.upper()
            .str.strip()
            .replace({

                "":
                    pd.NA,

                "NAN":
                    pd.NA,

                "NONE":
                    pd.NA,

                "<NA>":
                    pd.NA,

            })
        )

    for text_column in [
        "issuer_name",
        "security_title",
        "investment_country",
    ]:

        if text_column not in dataframe.columns:

            dataframe[
                text_column
            ] = pd.NA

    fallback_name = (

        dataframe[
            "issuer_name"
        ].fillna("")

        + "|"

        + dataframe[
            "security_title"
        ].fillna("")

        + "|"

        + dataframe[
            "investment_country"
        ].fillna("")

    ).map(
        normalise_text
    )

    identifier_conditions = [

        dataframe[
            "isin"
        ].notna(),

        dataframe[
            "cusip"
        ].notna(),

        dataframe[
            "ticker"
        ].notna(),

    ]

    identifier_values = [

        "ISIN:"
        + dataframe[
            "isin"
        ].fillna(""),

        "CUSIP:"
        + dataframe[
            "cusip"
        ].fillna(""),

        "TICKER:"
        + dataframe[
            "ticker"
        ].fillna(""),

    ]

    dataframe[
        "security_id"
    ] = np.select(
        identifier_conditions,
        identifier_values,
        default=(
            "NAME:"
            + fallback_name
        ),
    )

    dataframe.loc[
        dataframe[
            "security_id"
        ].eq("NAME:"),
        "security_id",
    ] = pd.NA

    return dataframe


# ------------------------------------------------


# 17. FILTER EQUITY-LIKE HOLDINGS
# ------------------------------------------------

def filter_equity_holdings(
    holdings: pd.DataFrame,
) -> pd.DataFrame:
    """
    Remove obvious debt, cash, repo and derivative holdings.
    """

    if holdings.empty:
        return holdings.copy()

    if KEEP_NON_EQUITY_HOLDINGS:
        return holdings.copy()

    dataframe = holdings.copy()

    for column in [
        "asset_category",
        "issuer_type",
        "payoff_profile",
        "security_title",
    ]:

        if column not in dataframe.columns:

            dataframe[
                column
            ] = ""

    asset_text = (

        dataframe[
            "asset_category"
        ]
        .fillna("")
        .astype(str)
        .str.upper()
    )

    issuer_text = (

        dataframe[
            "issuer_type"
        ]
        .fillna("")
        .astype(str)
        .str.upper()
    )

    payoff_text = (

        dataframe[
            "payoff_profile"
        ]
        .fillna("")
        .astype(str)
        .str.upper()
    )

    title_text = (

        dataframe[
            "security_title"
        ]
        .fillna("")
        .astype(str)
        .str.upper()
    )

    obvious_non_equity = (

        asset_text.str.contains(
            r"DEBT|BOND|LOAN|DERIV|SWAP|OPTION|"
            r"FUTURE|REPO|CASH",
            regex=True,
            na=False,
        )

        | issuer_text.str.contains(
            r"GOVERNMENT|MUNICIPAL",
            regex=True,
            na=False,
        )

        | title_text.str.contains(
            r"\bBOND\b|\bNOTE\b|\bTREASURY\b|"
            r"\bSWAP\b|\bOPTION\b|\bFUTURE\b|"
            r"\bREPURCHASE AGREEMENT\b",
            regex=True,
            na=False,
        )

        | payoff_text.eq(
            "SHORT"
        )
    )

    has_identifier = (

        dataframe[
            "ticker"
        ].notna()

        | dataframe[
            "isin"
        ].notna()

        | dataframe[
            "cusip"
        ].notna()

        | dataframe[
            "security_id"
        ].notna()
    )

    return dataframe[
        (~obvious_non_equity)
        & has_identifier
    ].copy()


# ------------------------------------------------


# 18. RESOLVE AMENDED FILINGS
# ------------------------------------------------

def resolve_amendments(
    holdings: pd.DataFrame,
) -> pd.DataFrame:
    """
    Select the latest complete public filing version for each ETF snapshot.

    Selection occurs at filing/accession level, not security-row level. This
    prevents an original filing and a later amendment being blended into one
    synthetic portfolio.
    """
    if holdings.empty:
        return holdings.copy()

    required = ["etf", "snapshot_date", "filing_date", "accession_number"]
    missing = [column for column in required if column not in holdings.columns]
    if missing:
        raise KeyError(f"Cannot resolve amendments; missing columns: {missing}")

    filing_columns = [
        column for column in [
            "etf",
            "series_id",
            "snapshot_date",
            "filing_date",
            "accession_number",
            "submission_type",
        ]
        if column in holdings.columns
    ]

    filing_versions = (
        holdings[filing_columns]
        .drop_duplicates()
        .sort_values(["etf", "snapshot_date", "filing_date", "accession_number"])
    )

    group_keys = [
        column for column in ["etf", "series_id", "snapshot_date"]
        if column in filing_versions.columns
    ]

    selected = (
        filing_versions
        .drop_duplicates(group_keys, keep="last")
        [group_keys + ["accession_number"]]
    )

    return holdings.merge(
        selected,
        on=group_keys + ["accession_number"],
        how="inner",
        validate="many_to_one",
    )


# ------------------------------------------------


# 19. BUILD CONSTITUENT SNAPSHOTS
# ------------------------------------------------

def build_constituent_snapshots(
    holdings: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create one row per ETF, filing snapshot and security.
    """

    if holdings.empty:
        return holdings.copy()

    group_columns = [

        column

        for column in [
            "etf",
            "series_id",
            "series_name",
            "snapshot_date",
            "filing_date",
            "effective_date",
            "accession_number",
            "security_id",
            "ticker",
            "cusip",
            "isin",
            "issuer_name",
            "security_title",
            "issuer_lei",
            "other_identifier",
            "other_identifier_description",
            "investment_country",
            "currency",
            "asset_category",
            "issuer_type",
        ]

        if column in holdings.columns
    ]

    aggregation = {}

    for numeric_column in [
        "market_value_usd",
        "market_value",
        "weight",
        "balance",
    ]:

        if numeric_column in holdings.columns:

            aggregation[
                numeric_column
            ] = "sum"

    if aggregation:

        snapshots = (

            holdings
            .groupby(
                group_columns,
                dropna=False,
                as_index=False,
            )
            .agg(
                aggregation
            )
        )

    else:

        snapshots = (

            holdings[
                group_columns
            ]
            .drop_duplicates()
        )

    snapshots[
        "is_constituent"
    ] = 1

    return snapshots.sort_values(
        [
            "effective_date",
            "etf",
            "weight",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )


# ------------------------------------------------


# 20. BUILD LATEST PUBLIC SNAPSHOT
# ------------------------------------------------

def build_latest_public_snapshot(
    snapshots: pd.DataFrame,
) -> pd.DataFrame:
    """
    Return each ETF's latest publicly available constituent filing.
    """

    if snapshots.empty:
        return snapshots.copy()

    latest_dates = (

        snapshots
        .groupby(
            "etf",
            as_index=False,
        )[
            "effective_date"
        ]
        .max()
        .rename(
            columns={
                "effective_date":
                    "latest_effective_date"
            }
        )
    )

    output = snapshots.merge(
        latest_dates,
        on="etf",
        how="inner",
    )

    output = output[
        output[
            "effective_date"
        ].eq(
            output[
                "latest_effective_date"
            ]
        )
    ].copy()

    return output.sort_values(
        [
            "etf",
            "weight",
        ],
        ascending=[
            True,
            False,
        ],
    )


# ------------------------------------------------


# 21. BUILD COVERAGE REPORT
# ------------------------------------------------

def build_coverage_report(
    fund_filings: pd.DataFrame,
    snapshots: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create coverage diagnostics for the four ETFs.
    """

    rows = []

    for etf in TARGET_FUNDS:

        if fund_filings.empty:

            filing_subset = pd.DataFrame()

        else:

            filing_subset = fund_filings[
                fund_filings[
                    "etf"
                ].eq(etf)
            ]

        if snapshots.empty:

            holding_subset = pd.DataFrame()

        else:

            holding_subset = snapshots[
                snapshots[
                    "etf"
                ].eq(etf)
            ]

        rows.append({

            "etf":
                etf,

            "filings_found":
                (
                    filing_subset[
                        "ACCESSION_NUMBER"
                    ].nunique()

                    if (
                        not filing_subset.empty
                        and "ACCESSION_NUMBER"
                        in filing_subset.columns
                    )

                    else 0
                ),

            "series_found":
                (
                    filing_subset[
                        "SERIES_ID"
                    ].nunique()

                    if (
                        not filing_subset.empty
                        and "SERIES_ID"
                        in filing_subset.columns
                    )

                    else 0
                ),

            "legal_names_found":
                (
                    filing_subset[
                        "SERIES_NAME_STANDARD"
                    ].nunique()

                    if (
                        not filing_subset.empty
                        and "SERIES_NAME_STANDARD"
                        in filing_subset.columns
                    )

                    else 0
                ),

            "first_snapshot_date":
                (
                    holding_subset[
                        "snapshot_date"
                    ].min()

                    if not holding_subset.empty

                    else pd.NaT
                ),

            "last_snapshot_date":
                (
                    holding_subset[
                        "snapshot_date"
                    ].max()

                    if not holding_subset.empty

                    else pd.NaT
                ),

            "first_effective_date":
                (
                    holding_subset[
                        "effective_date"
                    ].min()

                    if not holding_subset.empty

                    else pd.NaT
                ),

            "last_effective_date":
                (
                    holding_subset[
                        "effective_date"
                    ].max()

                    if not holding_subset.empty

                    else pd.NaT
                ),

            "unique_securities":
                (
                    holding_subset[
                        "security_id"
                    ].nunique()

                    if not holding_subset.empty

                    else 0
                ),

            "snapshot_rows":
                len(
                    holding_subset
                ),

        })

    return pd.DataFrame(
        rows
    )


# ------------------------------------------------


# 22. BUILD LEGAL-NAME HISTORY
# ------------------------------------------------

def build_legal_name_history(
    fund_filings: pd.DataFrame,
) -> pd.DataFrame:
    """
    Retain legal series-name changes, including the CARZ rename.
    """

    if fund_filings.empty:
        return pd.DataFrame()

    columns = [

        column

        for column in [
            "etf",
            "SERIES_ID",
            "SERIES_NAME_STANDARD",
            "matched_alias",
            "REPORT_DATE",
            "FILING_DATE",
            "ACCESSION_NUMBER",
            "dataset_year",
            "dataset_quarter",
        ]

        if column in fund_filings.columns
    ]

    output = (

        fund_filings[
            columns
        ]
        .drop_duplicates()
    )

    for date_column in [
        "REPORT_DATE",
        "FILING_DATE",
    ]:

        if date_column in output.columns:

            output[
                date_column
            ] = pd.to_datetime(
                output[
                    date_column
                ],
                errors="coerce",
            )

    sort_columns = [

        column

        for column in [
            "etf",
            "FILING_DATE",
            "REPORT_DATE",
        ]

        if column in output.columns
    ]

    return output.sort_values(
        sort_columns
    )


# ------------------------------------------------


In [ ]:

# ------------------------------------------------
# 23. BUILD POINT-IN-TIME AVAILABILITY INTERVALS
# ------------------------------------------------

def build_snapshot_index(
    constituent_snapshots: pd.DataFrame,
) -> pd.DataFrame:
    """
    One row per distinct public ETF snapshot.

    The index is the bridge between observed SEC filings and any later
    month-end universe-selection calendar.
    """
    columns = [
        "etf",
        "series_id",
        "series_name",
        "accession_number",
        "submission_type",
        "snapshot_date",
        "filing_date",
        "effective_date",
        "available_date",
        "dataset_year",
        "dataset_quarter",
    ]

    if constituent_snapshots.empty:
        return pd.DataFrame(columns=columns + ["next_available_date", "is_latest_public_snapshot"])

    df = constituent_snapshots.copy()

    # Preserve the earlier block's naming while exposing an explicit
    # downstream point-in-time field.
    if "available_date" not in df.columns:
        df["available_date"] = pd.to_datetime(
            df.get("effective_date", df.get("filing_date")),
            errors="coerce",
        )

    keep = [column for column in columns if column in df.columns]
    index_df = (
        df[keep]
        .drop_duplicates()
        .sort_values(["etf", "available_date", "snapshot_date", "accession_number"])
        .reset_index(drop=True)
    )

    index_df["next_available_date"] = (
        index_df.groupby("etf", dropna=False)["available_date"].shift(-1)
    )
    index_df["is_latest_public_snapshot"] = index_df["next_available_date"].isna()

    return index_df


def build_constituent_intervals(
    constituent_snapshots: pd.DataFrame,
    snapshot_index: pd.DataFrame,
) -> pd.DataFrame:
    """
    Attach an availability interval to every constituent row.

    Later modules can perform an as-of join using:
        available_date <= model_date < next_available_date
    """
    if constituent_snapshots.empty:
        return constituent_snapshots.copy()

    join_keys = [
        key for key in [
            "etf",
            "accession_number",
            "snapshot_date",
            "effective_date",
        ]
        if key in constituent_snapshots.columns and key in snapshot_index.columns
    ]

    interval_columns = join_keys + [
        column for column in [
            "available_date",
            "next_available_date",
            "is_latest_public_snapshot",
        ]
        if column in snapshot_index.columns
    ]

    result = constituent_snapshots.merge(
        snapshot_index[interval_columns].drop_duplicates(join_keys),
        on=join_keys,
        how="left",
        validate="many_to_one",
        suffixes=("", "_interval"),
    )

    # Avoid duplicate available_date after merging.
    if "available_date_interval" in result.columns:
        if "available_date" in result.columns:
            result["available_date"] = result["available_date"].fillna(
                result["available_date_interval"]
            )
            result = result.drop(columns=["available_date_interval"])
        else:
            result = result.rename(columns={"available_date_interval": "available_date"})

    return result


def build_security_master_seed(
    constituent_intervals: pd.DataFrame,
) -> pd.DataFrame:
    """
    Deduplicated identifier observations for Block 2's security master.

    This is deliberately a seed rather than a resolved master: identifiers
    can change, tickers are not permanent, and issuer/security identity must
    be resolved in the dedicated security-master block.
    """
    output_columns = [
        "etf",
        "snapshot_date",
        "available_date",
        "issuer_name",
        "security_title",
        "ticker",
        "isin",
        "cusip",
        "issuer_lei",
        "other_identifier",
        "other_identifier_description",
        "investment_country",
        "currency",
        "source_system",
        "source_accession_number",
    ]

    if constituent_intervals.empty:
        return pd.DataFrame(columns=output_columns)

    df = constituent_intervals.copy()
    df["source_system"] = "SEC_NPORT"
    df["source_accession_number"] = df.get("accession_number", pd.NA)

    keep = [column for column in output_columns if column in df.columns]
    seed = (
        df[keep]
        .drop_duplicates()
        .sort_values(
            [column for column in ["available_date", "etf", "issuer_name", "security_title"] if column in keep]
        )
        .reset_index(drop=True)
    )

    for column in output_columns:
        if column not in seed.columns:
            seed[column] = pd.NA

    return seed[output_columns]


def reconstruct_constituents_as_of(
    constituent_intervals: pd.DataFrame,
    as_of_date: str | pd.Timestamp,
    etfs: Iterable[str] | None = None,
) -> pd.DataFrame:
    """
    Return the latest legally available constituent snapshot for each ETF
    at a requested historical date.
    """
    if constituent_intervals.empty:
        return constituent_intervals.copy()

    as_of = pd.Timestamp(as_of_date).normalize()
    df = constituent_intervals.copy()

    mask = (
        df["available_date"].le(as_of)
        & (
            df["next_available_date"].isna()
            | df["next_available_date"].gt(as_of)
        )
    )

    if etfs is not None:
        requested = {str(etf).upper() for etf in etfs}
        mask &= df["etf"].astype("string").str.upper().isin(requested)

    result = df.loc[mask].copy()
    result["as_of_date"] = as_of
    return result.reset_index(drop=True)


# ------------------------------------------------
# 24. DOWNSTREAM DATAFRAME CONTRACT
# ------------------------------------------------

BLOCK_1_OUTPUT_COLUMNS = pd.DataFrame(
    [
        ("etf_filing_history_df", "One row per target N-PORT filing", "filing/accession grain"),
        ("etf_holdings_raw_df", "Raw target holdings after SEC table joins", "holding grain"),
        ("etf_holdings_standardised_df", "Stable cross-quarter field names and dtypes", "holding grain"),
        ("etf_constituent_snapshots_df", "Observed public equity-like snapshots", "ETF/snapshot/security grain"),
        ("etf_snapshot_index_df", "One row per public ETF snapshot", "ETF/snapshot grain"),
        ("etf_constituent_intervals_df", "Point-in-time validity intervals", "ETF/interval/security grain"),
        ("security_master_seed_df", "Unresolved identifier observations for Block 2", "identifier observation grain"),
        ("etf_coverage_report_df", "Coverage and quality summary", "ETF grain"),
    ],
    columns=["dataframe_name", "purpose", "grain"],
)


In [ ]:

# ------------------------------------------------
# 25. EXECUTE BLOCK 1
# ------------------------------------------------

all_fund_filings = []
all_raw_holdings = []
download_log_rows = []

for dataset in tqdm(
    available_datasets.itertuples(index=False),
    total=len(available_datasets),
    desc="Processing N-PORT quarters",
):
    try:
        zip_path = download_quarter(
            year=int(dataset.year),
            quarter=int(dataset.quarter),
            url=str(dataset.url),
        )

        quarter_filings, quarter_holdings = process_quarter(
            zip_path=zip_path,
            year=int(dataset.year),
            quarter=int(dataset.quarter),
        )

        if not quarter_filings.empty:
            all_fund_filings.append(quarter_filings)

        if not quarter_holdings.empty:
            all_raw_holdings.append(quarter_holdings)

        download_log_rows.append({
            "dataset_year": int(dataset.year),
            "dataset_quarter": int(dataset.quarter),
            "url": str(dataset.url),
            "status": "success",
            "target_filing_rows": len(quarter_filings),
            "target_holding_rows": len(quarter_holdings),
            "error": pd.NA,
        })

    except Exception as exc:
        download_log_rows.append({
            "dataset_year": int(dataset.year),
            "dataset_quarter": int(dataset.quarter),
            "url": str(dataset.url),
            "status": "failed",
            "target_filing_rows": 0,
            "target_holding_rows": 0,
            "error": f"{type(exc).__name__}: {exc}",
        })

etf_filing_history_df = (
    pd.concat(all_fund_filings, ignore_index=True)
    if all_fund_filings else pd.DataFrame()
)

etf_holdings_raw_df = (
    pd.concat(all_raw_holdings, ignore_index=True)
    if all_raw_holdings else pd.DataFrame()
)

nport_download_log_df = pd.DataFrame(download_log_rows)

etf_holdings_standardised_df = standardise_holdings(etf_holdings_raw_df)
etf_equity_holdings_df = filter_equity_holdings(etf_holdings_standardised_df)

# IMPORTANT: this function resolves duplicate/amended filings at the observed
# snapshot level. The original filing rows remain available in
# etf_filing_history_df and etf_holdings_standardised_df for auditability.
etf_latest_snapshot_versions_df = resolve_amendments(etf_equity_holdings_df)

etf_constituent_snapshots_df = build_constituent_snapshots(
    etf_latest_snapshot_versions_df
)

if not etf_constituent_snapshots_df.empty:
    etf_constituent_snapshots_df["available_date"] = pd.to_datetime(
        etf_constituent_snapshots_df["effective_date"],
        errors="coerce",
    )

etf_snapshot_index_df = build_snapshot_index(etf_constituent_snapshots_df)
etf_constituent_intervals_df = build_constituent_intervals(
    etf_constituent_snapshots_df,
    etf_snapshot_index_df,
)
security_master_seed_df = build_security_master_seed(
    etf_constituent_intervals_df
)

latest_public_constituents_df = build_latest_public_snapshot(
    etf_constituent_snapshots_df
)

etf_coverage_report_df = build_coverage_report(
    etf_filing_history_df,
    etf_constituent_snapshots_df,
)

etf_legal_name_history_df = build_legal_name_history(
    etf_filing_history_df
)

# Quality-control table at ETF/snapshot grain.
if not etf_constituent_snapshots_df.empty:
    etf_snapshot_quality_df = (
        etf_constituent_snapshots_df
        .assign(
            missing_ticker=lambda x: x["ticker"].isna(),
            missing_primary_id=lambda x: x["isin"].isna() & x["cusip"].isna(),
            negative_weight=lambda x: x["weight"].lt(0),
            filing_before_snapshot=lambda x: x["filing_date"].lt(x["snapshot_date"]),
        )
        .groupby(
            ["etf", "snapshot_date", "available_date", "accession_number"],
            as_index=False,
            dropna=False,
        )
        .agg(
            constituent_rows=("security_id", "size"),
            retained_equity_weight_total=("weight", "sum"),
            missing_ticker_rows=("missing_ticker", "sum"),
            missing_primary_id_rows=("missing_primary_id", "sum"),
            negative_weight_rows=("negative_weight", "sum"),
            filing_before_snapshot_rows=("filing_before_snapshot", "sum"),
        )
    )
else:
    etf_snapshot_quality_df = pd.DataFrame()

block_1_data = {
    "etf_registry_df": fund_aliases,
    "nport_dataset_catalog_df": available_datasets,
    "nport_download_log_df": nport_download_log_df,
    "etf_filing_history_df": etf_filing_history_df,
    "etf_holdings_raw_df": etf_holdings_raw_df,
    "etf_holdings_standardised_df": etf_holdings_standardised_df,
    "etf_equity_holdings_df": etf_equity_holdings_df,
    "etf_latest_snapshot_versions_df": etf_latest_snapshot_versions_df,
    "etf_constituent_snapshots_df": etf_constituent_snapshots_df,
    "etf_snapshot_index_df": etf_snapshot_index_df,
    "etf_constituent_intervals_df": etf_constituent_intervals_df,
    "latest_public_constituents_df": latest_public_constituents_df,
    "security_master_seed_df": security_master_seed_df,
    "etf_coverage_report_df": etf_coverage_report_df,
    "etf_legal_name_history_df": etf_legal_name_history_df,
    "etf_snapshot_quality_df": etf_snapshot_quality_df,
    "block_1_output_contract_df": BLOCK_1_OUTPUT_COLUMNS,
}

print("Block 1 reconstruction complete. DataFrames are ready for persistence.")
display(BLOCK_1_OUTPUT_COLUMNS)
display(etf_coverage_report_df)
display(etf_snapshot_quality_df.head(20))


Processing N-PORT quarters:   0%|          | 0/27 [00:00<?, ?it/s]

2019 Q4:   0%|          | 0.00/240M [00:00<?, ?B/s]

2020 Q1:   0%|          | 0.00/340M [00:00<?, ?B/s]

2020 Q2:   0%|          | 0.00/333M [00:00<?, ?B/s]

2020 Q3:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Block 1 reconstruction complete. DataFrames are ready for persistence.


/tmp/ipykernel_2493/1180521533.py:1012: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ] = pd.to_datetime(
/tmp/ipykernel_2493/1180521533.py:1012: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ] = pd.to_datetime(


,dataframe_name,purpose,grain
0,etf_filing_history_df,One row per target N-PORT filing,filing/accession grain
1,etf_holdings_raw_df,Raw target holdings after SEC table joins,holding grain
2,etf_holdings_standardised_df,Stable cross-quarter field names and dtypes,holding grain
3,etf_constituent_snapshots_df,Observed public equity-like snapshots,ETF/snapshot/security grain
4,etf_snapshot_index_df,One row per public ETF snapshot,ETF/snapshot grain
5,etf_constituent_intervals_df,Point-in-time validity intervals,ETF/interval/security grain
6,security_master_seed_df,Unresolved identifier observations for Block 2,identifier observation grain
7,etf_coverage_report_df,Coverage and quality summary,ETF grain


,etf,filings_found,series_found,legal_names_found,first_snapshot_date,last_snapshot_date,first_effective_date,last_effective_date,unique_securities,snapshot_rows
0,DRIV,27,1,1,2019-11-30,2026-02-28,2020-01-23,2026-04-29,160,1992
1,CARZ,28,1,2,2019-09-30,2026-03-31,2019-11-19,2026-05-21,194,2066
2,IDRV,27,1,1,2019-10-31,2026-04-30,2019-12-26,2026-06-25,254,2208
3,KARS,31,1,1,2019-09-30,2026-03-31,2019-11-27,2026-05-29,221,1762


,etf,snapshot_date,available_date,accession_number,constituent_rows,retained_equity_weight_total,missing_ticker_rows,missing_primary_id_rows,negative_weight_rows,filing_before_snapshot_rows
0,CARZ,2019-09-30,2019-11-19,0001752724-19-167449,34,1.003924,1,0,0,0
1,CARZ,2019-12-31,2020-02-26,0001752724-20-036089,34,1.004969,1,0,0,0
2,CARZ,2020-03-31,2020-05-20,0001752724-20-097059,33,1.004183,0,0,0,0
3,CARZ,2020-06-30,2020-08-26,0001752724-20-173911,33,0.997521,0,0,0,0
4,CARZ,2020-09-30,2020-11-23,0001752724-20-241236,33,0.99136,0,0,0,0
5,CARZ,2020-12-31,2021-02-24,0001752724-21-036442,33,0.998061,0,0,0,0
6,CARZ,2021-03-31,2021-05-20,0001752724-21-102522,33,0.996274,0,0,0,0
7,CARZ,2021-06-30,2021-08-25,0001752724-21-185370,34,0.999874,0,0,0,0
8,CARZ,2021-09-30,2021-11-18,0001752724-21-245447,35,1.017516,1,0,0,0
9,CARZ,2021-12-31,2022-02-24,0001752724-22-043482,34,1.003389,1,0,0,0


In [ ]:
# 26. PERSIST BLOCK 1 OUTPUTS FOR DOWNSTREAM NOTEBOOKS
# ------------------------------------------------

# These are the formal hand-off tables required by Block 2 and later modules.
# Large diagnostic/raw tables are included where useful, but downstream blocks
# should rely primarily on the security-master seed and point-in-time intervals.

if "block_1_data" not in globals() or not isinstance(block_1_data, dict):
    raise RuntimeError(
        "block_1_data is unavailable. Run Step 25 before Step 26."
    )

required_handoff_names = {
    "security_master_seed_df",
    "etf_constituent_intervals_df",
    "etf_constituent_snapshots_df",
    "etf_snapshot_index_df",
}

missing_bundle_outputs = required_handoff_names.difference(block_1_data)

if missing_bundle_outputs:
    raise RuntimeError(
        "Block 1 output bundle is missing required hand-off tables: "
        f"{sorted(missing_bundle_outputs)}"
    )

# The raw joined holdings table can be large, so it is tagged separately in
# the manifest. Every other declared Block 1 output is persisted normally.
BLOCK_1_OPTIONAL_LARGE_TABLES = {
    name: dataframe
    for name, dataframe in block_1_data.items()
    if name == "etf_holdings_raw_df"
}

BLOCK_1_PERSISTED_TABLES = {
    name: dataframe
    for name, dataframe in block_1_data.items()
    if name not in BLOCK_1_OPTIONAL_LARGE_TABLES
}

# Defensive validation before writing any files.
invalid_outputs = {
    name: type(dataframe).__name__
    for name, dataframe in {
        **BLOCK_1_PERSISTED_TABLES,
        **BLOCK_1_OPTIONAL_LARGE_TABLES,
    }.items()
    if not isinstance(dataframe, pd.DataFrame)
}

if invalid_outputs:
    raise TypeError(
        "All Block 1 outputs must be pandas DataFrames. Invalid entries: "
        f"{invalid_outputs}"
    )


def make_parquet_safe(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Return a Parquet-safe copy while preserving pandas datetime and numeric types.

    Mixed Python object columns are converted to pandas StringDtype. This avoids
    pyarrow failures caused by columns containing inconsistent Python types.
    """
    output = dataframe.copy()

    for column in output.columns:
        if output[column].dtype == "object":
            non_missing = output[column].dropna()

            if not non_missing.empty:
                observed_types = non_missing.map(type).nunique()

                if observed_types > 1:
                    output[column] = output[column].astype("string")

    return output


def persist_dataframe(
    name: str,
    dataframe: pd.DataFrame,
    output_dir: Path,
    *,
    overwrite: bool = True,
) -> dict:
    """
    Persist one DataFrame as Parquet and return manifest metadata.
    """
    output_path = output_dir / f"{name}.parquet"

    if output_path.exists() and not overwrite:
        raise FileExistsError(
            f"Refusing to overwrite existing output: {output_path}"
        )

    safe_df = make_parquet_safe(dataframe)

    safe_df.to_parquet(
        output_path,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )

    return {
        "table_name": name,
        "path": str(output_path),
        "row_count": int(len(safe_df)),
        "column_count": int(len(safe_df.columns)),
        "columns": list(map(str, safe_df.columns)),
        "file_size_bytes": int(output_path.stat().st_size),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }


def load_block_1_outputs(
    output_dir: Path = BLOCK_1_OUTPUT_DIR,
) -> dict[str, pd.DataFrame]:
    """
    Load all persisted Block 1 Parquet tables into a dictionary.

    This helper can be copied into Block 2 or replaced by its own loader.
    """
    manifest_path = output_dir / "block_1_manifest.json"

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Block 1 manifest not found: {manifest_path}"
        )

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    loaded = {}

    for table in manifest["tables"]:
        table_path = Path(table["path"])

        if not table_path.exists():
            raise FileNotFoundError(
                f"Manifest table is missing: {table_path}"
            )

        loaded[table["table_name"]] = pd.read_parquet(table_path)

    return loaded


if PERSIST_BLOCK_1_OUTPUTS:

    manifest_rows = []

    for table_name, dataframe in BLOCK_1_PERSISTED_TABLES.items():
        manifest_rows.append(
            persist_dataframe(
                table_name,
                dataframe,
                BLOCK_1_OUTPUT_DIR,
                overwrite=OVERWRITE_PERSISTED_OUTPUTS,
            )
        )

    # Persist the raw table as well, but tag it as optional in the manifest.
    for table_name, dataframe in BLOCK_1_OPTIONAL_LARGE_TABLES.items():
        metadata = persist_dataframe(
            table_name,
            dataframe,
            BLOCK_1_OUTPUT_DIR,
            overwrite=OVERWRITE_PERSISTED_OUTPUTS,
        )
        metadata["optional_large_table"] = True
        manifest_rows.append(metadata)

    manifest = {
        "block": 1,
        "block_name": "ETF constituent retrieval and point-in-time holdings reconstruction",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "output_directory": str(BLOCK_1_OUTPUT_DIR),
        "source": "SEC Form N-PORT structured-data bulk files",
        "start_year": START_YEAR,
        "start_quarter": START_QUARTER,
        "end_year": END_YEAR,
        "end_quarter": END_QUARTER,
        "target_funds": list(TARGET_FUNDS.keys()),
        "tables": manifest_rows,
    }

    with BLOCK_1_MANIFEST_PATH.open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)

    block_1_persistence_report_df = pd.DataFrame(manifest_rows)

    print("Block 1 outputs persisted successfully.")
    print("Manifest:", BLOCK_1_MANIFEST_PATH)
    display(
        block_1_persistence_report_df[
            [
                "table_name",
                "row_count",
                "column_count",
                "file_size_bytes",
                "path",
            ]
        ]
    )

else:

    block_1_persistence_report_df = pd.DataFrame()

    print(
        "PERSIST_BLOCK_1_OUTPUTS is False. "
        "Outputs remain available only in the current runtime."
    )


# ------------------------------------------------
# 27. PERSISTENCE VALIDATION
# ------------------------------------------------

if PERSIST_BLOCK_1_OUTPUTS:

    persisted_test = load_block_1_outputs()

    required_handoff_tables = {
        "security_master_seed_df",
        "etf_constituent_intervals_df",
        "etf_constituent_snapshots_df",
        "etf_snapshot_index_df",
    }

    missing_handoff_tables = required_handoff_tables.difference(
        persisted_test.keys()
    )

    if missing_handoff_tables:
        raise RuntimeError(
            "Persistence validation failed. Missing tables: "
            f"{sorted(missing_handoff_tables)}"
        )

    for required_name in required_handoff_tables:
        original_rows = len(BLOCK_1_PERSISTED_TABLES[required_name])
        reloaded_rows = len(persisted_test[required_name])

        if original_rows != reloaded_rows:
            raise RuntimeError(
                f"Row-count mismatch for {required_name}: "
                f"{original_rows} original versus {reloaded_rows} reloaded."
            )

    print(
        "Persistence validation passed. "
        "Block 2 can now load the Parquet outputs without rerunning Block 1."
    )

Block 1 outputs persisted successfully.
Manifest: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_1/block_1_manifest.json


,table_name,row_count,column_count,file_size_bytes,path
0,etf_registry_df,21,3,2972,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
1,nport_dataset_catalog_df,27,3,2802,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
2,nport_download_log_df,27,7,5266,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
3,etf_filing_history_df,113,11,10561,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
4,etf_holdings_standardised_df,8550,41,567586,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
5,etf_equity_holdings_df,8490,41,563696,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
6,etf_latest_snapshot_versions_df,8040,41,549903,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
7,etf_constituent_snapshots_df,8028,26,396008,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
8,etf_snapshot_index_df,107,10,12603,/content/drive/MyDrive/Colab Notebooks/00 A1 A...
9,etf_constituent_intervals_df,8028,28,398621,/content/drive/MyDrive/Colab Notebooks/00 A1 A...


Persistence validation passed. Block 2 can now load the Parquet outputs without rerunning Block 1.


## Downstream usage

After Block 1 completes, Block 2 should load the persisted files from:

```text
data/interim/block_1/
```

The most important hand-off tables are:

```python
security_master_seed_df
etf_constituent_intervals_df
etf_constituent_snapshots_df
etf_snapshot_index_df
```

A downstream notebook can load all outputs using the manifest:

```python
from pathlib import Path
import json
import pandas as pd

block_1_dir = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "00 A1 Auto Factor Strategy/data/interim/block_1"
)

with (block_1_dir / "block_1_manifest.json").open("r") as file:
    manifest = json.load(file)

block_1_data = {
    item["table_name"]: pd.read_parquet(item["path"])
    for item in manifest["tables"]
}
```

Block 2 should **not** execute the Block 1 notebook. It should read these persisted tables directly.

### As-of reconstruction example

```python
constituents_2021_06_30 = reconstruct_constituents_as_of(
    etf_constituent_intervals_df,
    "2021-06-30",
)
```

This returns only the latest ETF snapshot that had become publicly available by the requested date.
